# TI-DPO-compatible frozen generation suite

This Kaggle notebook is the **generation stage only**. It runs local Hugging Face causal
language models over MMLU, GSM8K, full GPQA Main, HumanEval, TruthfulQA MC2, and IFEval,
then saves durable raw generations or every candidate log-likelihood needed for later
offline scoring.

It never computes benchmark scores, calls a judge/API, or executes HumanEval completions.
Attach model directories as Kaggle Inputs, enable a GPU accelerator (T4 x2 recommended),
edit only the configuration cell below, and run all cells.


## 1. Configuration

This is the only cell intended for routine editing. Add or remove model dictionaries.
A per-model `apply_chat_template` boolean may override the global setting. Optional advanced
fields are `chat_template` (the name of an official tokenizer template when several exist),
`trust_remote_code`, `batch_size`, and `max_batch_size`.


In [ ]:
MODELS = [
    {
        "name": "<MODEL_NAME_1>",
        "path": "/kaggle/input/<MODEL_PATH_1>",
    },
    {
        "name": "<MODEL_NAME_2>",
        "path": "/kaggle/input/<MODEL_PATH_2>",
    },

    # ADD MORE MODELS HERE
]

OUTPUT_ROOT = "/kaggle/working/tidpo_generations"
HF_TOKEN_SECRET_NAME = "HF_TOKEN"
SUITE_NAME = "tidpo_compatible_v1"

APPLY_CHAT_TEMPLATE = True
SEED = 42

# Conservative defaults for T4 memory. The harness auto-tunes request batches.
DEFAULT_BATCH_SIZE = "auto:4"
DEFAULT_MAX_BATCH_SIZE = 16
SAMPLE_CHUNK_SIZE = 16
FLUSH_EVERY = 8


## 2. Dependency installation

The evaluator logic is pinned to lm-evaluation-harness v0.4.9.2 at an exact Git commit.
Core userspace dependencies are pinned as well. Kaggle's CUDA-enabled PyTorch build is
intentionally retained instead of being replaced; its exact version is recorded later.
Internet access must be enabled for this installation and the benchmark downloads.


In [ ]:
import subprocess
import sys

LM_EVAL_COMMIT = "ad3f4d0cad1cfcdb815f1e795f7947e49ed9f2e9"
LM_EVAL_VERSION = "0.4.9.2"

PINNED_PACKAGES = [
    "transformers==4.57.1",
    "accelerate==1.11.0",
    "datasets==4.4.1",
    "evaluate==0.4.6",
    "huggingface-hub==0.36.0",
    "tokenizers==0.22.1",
    "safetensors==0.6.2",
    "peft==0.17.1",
    "sentencepiece==0.2.1",
    "jsonlines==4.0.0",
    "langdetect==1.0.9",
    "immutabledict==4.2.1",
    "nltk==3.9.1",
    "rouge-score==0.1.2",
    "sacrebleu==2.5.1",
    "sqlitedict==2.1.0",
    "word2number==1.1",
    "more-itertools==10.6.0",
    "zstandard==0.23.0",
    "dill==0.4.0",
    "pytablewriter==1.2.1",
    "tqdm-multiprocess==0.0.11",
    "numexpr==2.10.2",
    "pybind11==2.13.6",
    "scikit-learn==1.6.1",
]

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check", "--upgrade"]
    + PINNED_PACKAGES
)
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--disable-pip-version-check",
        "--no-deps",
        "--force-reinstall",
        f"git+https://github.com/EleutherAI/lm-evaluation-harness.git@{LM_EVAL_COMMIT}",
    ]
)
print("Pinned generation environment installed:")
print("\n".join(f"  - {package}" for package in PINNED_PACKAGES))
print(f"  - lm-evaluation-harness @ {LM_EVAL_COMMIT}")


## 3. Environment diagnostics

This cell imports the installed stack, freezes random seeds, creates workspace-local caches,
and reports every visible GPU. T4 GPUs use FP16; BF16 is selected only when all visible GPUs
safely support it.


In [ ]:
import gc
import hashlib
import importlib.metadata
import json
import os
import random
import re
import shutil
import subprocess
import sys
import traceback
import warnings
from collections import defaultdict
from copy import deepcopy
from datetime import datetime, timezone
from pathlib import Path

OUTPUT_ROOT_PATH = Path(OUTPUT_ROOT).expanduser().resolve()
OUTPUT_ROOT_PATH.mkdir(parents=True, exist_ok=True)
CACHE_ROOT = (
    Path("/kaggle/working/.cache/tidpo_generation")
    / SUITE_NAME
    / f"lm-eval-{LM_EVAL_COMMIT[:12]}"
    / f"seed-{SEED}"
)
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(CACHE_ROOT / "huggingface")
os.environ["HF_DATASETS_CACHE"] = str(CACHE_ROOT / "huggingface/datasets")
os.environ["TRANSFORMERS_CACHE"] = str(CACHE_ROOT / "huggingface/transformers")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import accelerate
import datasets
import huggingface_hub
import lm_eval
import numpy as np
import torch
import transformers
from tqdm.auto import tqdm


def utc_now():
    return datetime.now(timezone.utc).isoformat()


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


seed_everything(SEED)
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. In Kaggle, select Settings > Accelerator > GPU T4 x2.")

GPU_INFO = []
for gpu_index in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(gpu_index)
    item = {
        "index": gpu_index,
        "name": props.name,
        "total_memory_bytes": int(props.total_memory),
        "compute_capability": [int(props.major), int(props.minor)],
    }
    GPU_INFO.append(item)
    print(
        f"GPU {gpu_index}: {props.name} | {props.total_memory / 2**30:.2f} GiB | "
        f"compute capability {props.major}.{props.minor}"
    )

if torch.cuda.device_count() < 2:
    warnings.warn("Only one GPU is visible. The notebook supports it, but Kaggle T4 x2 is recommended.")
elif all("T4" in item["name"] for item in GPU_INFO[:2]):
    print("Kaggle dual-T4 environment detected.")

all_bf16 = bool(
    torch.cuda.is_bf16_supported()
    and all(item["compute_capability"][0] >= 8 for item in GPU_INFO)
)
MODEL_DTYPE = torch.bfloat16 if all_bf16 else torch.float16
MODEL_DTYPE_NAME = "bfloat16" if all_bf16 else "float16"

PACKAGE_VERSIONS = {
    "python": sys.version,
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "transformers": transformers.__version__,
    "datasets": datasets.__version__,
    "accelerate": accelerate.__version__,
    "huggingface_hub": huggingface_hub.__version__,
    "lm_eval": importlib.metadata.version("lm_eval"),
    "lm_eval_commit": LM_EVAL_COMMIT,
    "lm_eval_module": str(Path(lm_eval.__file__).resolve()),
}
lm_eval_distribution = importlib.metadata.distribution("lm_eval")
direct_url_text = lm_eval_distribution.read_text("direct_url.json")
LM_EVAL_DIRECT_URL = json.loads(direct_url_text) if direct_url_text else {}
installed_lm_eval_commit = LM_EVAL_DIRECT_URL.get("vcs_info", {}).get("commit_id")
if PACKAGE_VERSIONS["lm_eval"] != LM_EVAL_VERSION or installed_lm_eval_commit != LM_EVAL_COMMIT:
    raise RuntimeError(
        "Pinned lm-evaluation-harness verification failed: "
        f"version={PACKAGE_VERSIONS['lm_eval']!r}, commit={installed_lm_eval_commit!r}"
    )
print(json.dumps(PACKAGE_VERSIONS, indent=2))
print("Inference dtype:", MODEL_DTYPE_NAME)


## 4. Authentication

GPQA is gated. Accept the dataset's terms on Hugging Face, add a Kaggle secret named
`HF_TOKEN`, and grant the notebook access to it. Missing authentication is reported without
ever printing the token.


In [ ]:
HF_TOKEN_AVAILABLE = False
try:
    from kaggle_secrets import UserSecretsClient

    hf_token = UserSecretsClient().get_secret(HF_TOKEN_SECRET_NAME)
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
        huggingface_hub.login(token=hf_token, add_to_git_credential=False)
        HF_TOKEN_AVAILABLE = True
        print(f"Hugging Face authentication loaded from Kaggle Secret {HF_TOKEN_SECRET_NAME!r}.")
    del hf_token
except Exception as exc:
    warnings.warn(
        f"Hugging Face token was not loaded ({type(exc).__name__}). "
        "Public tasks can run, but GPQA Main may fail until its terms are accepted and the secret is attached."
    )


## 5. Frozen benchmark configuration

The harness and every source dataset are pinned to immutable commits. `gpqa_main_zeroshot`
is explicitly guarded against Diamond. Generation limits are deliberate safety limits;
deterministic decoding uses one sequence with sampling disabled. The runtime protocol hash
also incorporates resolved task configs, source fingerprints, and expected sample IDs.


In [ ]:
DATASET_REVISIONS = {
    "mmlu": "c30699e8356da336a370243923dbaf21066bb9fe",
    "gsm8k": "740312add88f781978c0658806c59bc2815b9866",
    "gpqa": "633f5ee89ab8ad4522a9f850766b73f62147ffdd",
    "humaneval": "7dce6050a7d6d172f3cc5c32aa97f52fa1a2e544",
    "truthfulqa": "741b8276f2d1982aa3d5b832d3ee81ed3b896490",
    "ifeval": "966cd89545d6b6acfd7638bc708b98261ca58e84",
}

BENCHMARK_PROTOCOLS = {
    "mmlu": {
        "harness_task": "mmlu",
        "dataset_path": "cais/mmlu",
        "dataset_revision": DATASET_REVISIONS["mmlu"],
        "variant": "original full 57-subject MMLU",
        "output_type": "multiple_choice",
        "num_fewshot": 5,
        "generation_kwargs": None,
    },
    "gsm8k": {
        "harness_task": "gsm8k",
        "dataset_path": "openai/gsm8k",
        "dataset_revision": DATASET_REVISIONS["gsm8k"],
        "variant": "main/test",
        "output_type": "generate_until",
        "num_fewshot": 5,
        "generation_kwargs": {
            "until": ["Question:", "</s>", "<|im_end|>"],
            "do_sample": False,
            "temperature": 0.0,
            "max_gen_toks": 512,
        },
    },
    "gpqa": {
        "harness_task": "gpqa_main_zeroshot",
        "dataset_path": "Idavidrein/gpqa",
        "dataset_revision": DATASET_REVISIONS["gpqa"],
        "dataset_name": "gpqa_main",
        "variant": "FULL GPQA Main; never Diamond",
        "output_type": "multiple_choice",
        "num_fewshot": 0,
        "generation_kwargs": None,
    },
    "humaneval": {
        "harness_task": "humaneval_generation_only",
        "canonical_harness_task": "humaneval",
        "dataset_path": "openai/openai_humaneval",
        "dataset_revision": DATASET_REVISIONS["humaneval"],
        "variant": "OpenAI HumanEval all 164 problems; generation only",
        "output_type": "generate_until",
        "num_fewshot": 0,
        "generation_kwargs": {
            "until": ["\nclass", "\ndef", "\n#", "\nif", "\nprint"],
            "do_sample": False,
            "temperature": 0.0,
            "max_gen_toks": 1024,
        },
    },
    "truthfulqa": {
        "harness_task": "truthfulqa_mc2",
        "dataset_path": "truthful_qa",
        "dataset_revision": DATASET_REVISIONS["truthfulqa"],
        "dataset_name": "multiple_choice",
        "variant": "original TruthfulQA MC2",
        "output_type": "multiple_choice",
        "num_fewshot": 0,
        "generation_kwargs": None,
    },
    "ifeval": {
        "harness_task": "ifeval",
        "dataset_path": "google/IFEval",
        "dataset_revision": DATASET_REVISIONS["ifeval"],
        "variant": "original Google IFEval",
        "output_type": "generate_until",
        "num_fewshot": 0,
        "generation_kwargs": {
            "until": [],
            "do_sample": False,
            "temperature": 0.0,
            "max_gen_toks": 1280,
        },
    },
}
BENCHMARK_ORDER = ["mmlu", "gsm8k", "gpqa", "humaneval", "truthfulqa", "ifeval"]

BASE_PROTOCOL = {
    "suite": SUITE_NAME,
    "seed": SEED,
    "apply_chat_template_default": APPLY_CHAT_TEMPLATE,
    "fewshot_as_multiturn_when_chat": True,
    "lm_eval_version": LM_EVAL_VERSION,
    "lm_eval_commit": LM_EVAL_COMMIT,
    "benchmarks": BENCHMARK_PROTOCOLS,
    "human_eval_execution": False,
    "scoring_performed": False,
}


## 6. Stable-file and serialization helpers

JSONL writes are append-only and flushed frequently. On resume, a malformed trailing record
is truncated to the last valid byte; valid records are retained. Existing records from a
different model or protocol cause a hard error rather than being overwritten.


In [ ]:
def jsonable(value):
    if value is None or isinstance(value, (str, int, float, bool)):
        return value
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {str(key): jsonable(item) for key, item in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [jsonable(item) for item in value]
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, torch.dtype):
        return str(value)
    if hasattr(value, "to_dict"):
        return jsonable(value.to_dict())
    return str(value)


def canonical_json(value):
    return json.dumps(jsonable(value), sort_keys=True, separators=(",", ":"), ensure_ascii=False)


def sha256_text(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def atomic_write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(jsonable(payload), indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    os.replace(temporary, path)


def safe_slug(name):
    slug = re.sub(r"[^A-Za-z0-9._-]+", "_", name.strip()).strip("._")
    if not slug:
        raise ValueError(f"Model name does not produce a safe output directory: {name!r}")
    return slug


def repair_and_read_jsonl(path):
    path = Path(path)
    if not path.exists():
        return []
    records = []
    valid_offset = 0
    with path.open("rb") as handle:
        while True:
            line = handle.readline()
            if not line:
                break
            try:
                record = json.loads(line.decode("utf-8"))
            except Exception:
                warnings.warn(f"Truncating malformed JSONL tail in {path} at byte {valid_offset}.")
                break
            records.append(record)
            valid_offset = handle.tell()
    if valid_offset != path.stat().st_size:
        with path.open("r+b") as handle:
            handle.truncate(valid_offset)
    seen = set()
    for record in records:
        sample_id = record.get("sample_id")
        if not sample_id or sample_id in seen:
            raise RuntimeError(f"Missing or duplicate sample_id in {path}: {sample_id!r}")
        seen.add(sample_id)
    return records


def append_records(path, records):
    if not records:
        return
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8", newline="\n") as handle:
        for index, record in enumerate(records, start=1):
            handle.write(json.dumps(jsonable(record), ensure_ascii=False, separators=(",", ":")) + "\n")
            if index % FLUSH_EVERY == 0:
                handle.flush()
                os.fsync(handle.fileno())
        handle.flush()
        os.fsync(handle.fileno())


def chunks(items, size):
    for start in range(0, len(items), size):
        yield items[start : start + size]


def display_gpu_memory():
    for gpu_index in range(torch.cuda.device_count()):
        free_bytes, total_bytes = torch.cuda.mem_get_info(gpu_index)
        allocated = torch.cuda.memory_allocated(gpu_index)
        reserved = torch.cuda.memory_reserved(gpu_index)
        print(
            f"GPU {gpu_index} memory: free={free_bytes / 2**30:.2f} GiB, "
            f"allocated={allocated / 2**30:.2f} GiB, reserved={reserved / 2**30:.2f} GiB, "
            f"total={total_bytes / 2**30:.2f} GiB"
        )


## 7. Harness task loading and protocol fingerprint

The exact pinned harness tasks build prompts and inference requests. Dataset revisions are
injected before loading. GPQA's deterministic shuffle is performed once after seeding, and
its shuffled choice text and ground-truth mapping are exported per record. HumanEval uses an
equivalent metric-free harness config because v0.4.9.2's canonical utility executes a
`code_eval` smoke test at import time; that utility is deliberately never imported here.


In [ ]:
from lm_eval.evaluator_utils import get_task_list
from lm_eval.tasks import TaskManager, get_task_dict


def load_frozen_tasks():
    seed_everything(SEED)
    manager = TaskManager(verbosity="ERROR")
    loaded = {}
    for benchmark in BENCHMARK_ORDER:
        protocol = BENCHMARK_PROTOCOLS[benchmark]
        if benchmark == "humaneval":
            # Do not load the registered `humaneval` YAML: in pinned harness v0.4.9.2 its
            # imported metric utility calls code_eval.compute() immediately. This unregistered
            # task exactly preserves its dataset, prompt, reference, stops, and generation
            # settings while defining no metric and importing no executable-code evaluator.
            task_request = {
                "task": "humaneval_generation_only",
                "dataset_path": protocol["dataset_path"],
                "dataset_kwargs": {"revision": protocol["dataset_revision"]},
                "output_type": "generate_until",
                "test_split": "test",
                "doc_to_text": "{{prompt}}",
                "doc_to_target": "{{test}}\ncheck({{entry_point}})",
                "generation_kwargs": deepcopy(protocol["generation_kwargs"]),
                "repeats": 1,
                "num_fewshot": 0,
                "metric_list": [],
                "unsafe_code": False,
                "metadata": {
                    "version": 1.0,
                    "canonical_harness_task": "humaneval",
                    "generation_only": True,
                    "code_execution": False,
                },
            }
        else:
            task_request = {
                "task": protocol["harness_task"],
                "dataset_kwargs": {"revision": protocol["dataset_revision"]},
                "num_fewshot": protocol["num_fewshot"],
            }
        task_dict = get_task_dict([task_request], task_manager=manager)
        outputs = [item for item in get_task_list(task_dict) if item.task is not None]
        if not outputs:
            raise RuntimeError(f"Harness resolved no concrete tasks for {benchmark}.")
        for output in outputs:
            output.task.set_fewshot_seed(SEED)
            output.task._config.num_fewshot = protocol["num_fewshot"]
            if protocol["generation_kwargs"] is not None:
                output.task._config.generation_kwargs = deepcopy(protocol["generation_kwargs"])
        loaded[benchmark] = outputs
        print(f"Loaded {benchmark}: {len(outputs)} concrete harness task(s)")
    return loaded


TASK_OUTPUTS_BY_BENCHMARK = load_frozen_tasks()


def sample_id_for(task_name, doc_id):
    return f"{task_name}:{int(doc_id):06d}"


def expected_inventory(benchmark):
    inventory = []
    for output in TASK_OUTPUTS_BY_BENCHMARK[benchmark]:
        for doc_id, _doc in enumerate(output.task.eval_docs):
            inventory.append(
                {
                    "sample_id": sample_id_for(output.task_name, doc_id),
                    "task_name": output.task_name,
                    "doc_id": int(doc_id),
                }
            )
    return inventory


EXPECTED_INVENTORIES = {name: expected_inventory(name) for name in BENCHMARK_ORDER}
EXPECTED_IDS = {
    name: {item["sample_id"] for item in inventory}
    for name, inventory in EXPECTED_INVENTORIES.items()
}

mmlu_tasks = TASK_OUTPUTS_BY_BENCHMARK["mmlu"]
mmlu_subjects = sorted(item.task_name.removeprefix("mmlu_") for item in mmlu_tasks)
if len(mmlu_tasks) != 57 or len(set(mmlu_subjects)) != 57:
    raise RuntimeError(f"Expected all 57 original MMLU subjects; resolved {len(set(mmlu_subjects))}.")

gpqa_outputs = TASK_OUTPUTS_BY_BENCHMARK["gpqa"]
if len(gpqa_outputs) != 1:
    raise RuntimeError(f"Expected one GPQA Main task, found {len(gpqa_outputs)}.")
gpqa_task = gpqa_outputs[0].task
if gpqa_outputs[0].task_name != "gpqa_main_zeroshot" or gpqa_task.config.dataset_name != "gpqa_main":
    raise RuntimeError("GPQA guard failed: this notebook must use FULL gpqa_main, never gpqa_diamond.")

if len(EXPECTED_IDS["humaneval"]) != 164:
    raise RuntimeError(f"Expected 164 HumanEval problems; found {len(EXPECTED_IDS['humaneval'])}.")


def task_fingerprint(output):
    dataset = output.task.eval_docs
    return {
        "task_name": output.task_name,
        "task_version": (output.task.config.metadata or {}).get("version", output.version),
        "task_config": output.task.config.to_dict(),
        "dataset_fingerprint": getattr(dataset, "_fingerprint", None),
        "sample_count": len(dataset),
    }


RESOLVED_TASKS = {
    benchmark: [task_fingerprint(output) for output in outputs]
    for benchmark, outputs in TASK_OUTPUTS_BY_BENCHMARK.items()
}
FROZEN_PROTOCOL = {
    **BASE_PROTOCOL,
    "resolved_tasks": RESOLVED_TASKS,
    "expected_sample_ids_sha256": {
        name: sha256_text(canonical_json(sorted(ids))) for name, ids in EXPECTED_IDS.items()
    },
}
PROTOCOL_SHA256 = sha256_text(canonical_json(FROZEN_PROTOCOL))
PROTOCOL_PATH = OUTPUT_ROOT_PATH / "protocol.json"
if PROTOCOL_PATH.exists():
    existing_protocol = json.loads(PROTOCOL_PATH.read_text(encoding="utf-8"))
    if existing_protocol.get("protocol_sha256") != PROTOCOL_SHA256:
        raise RuntimeError(
            "Existing OUTPUT_ROOT was created with a different protocol or dataset fingerprint. "
            "Choose a new OUTPUT_ROOT; existing artifacts will not be overwritten."
        )
else:
    atomic_write_json(
        PROTOCOL_PATH,
        {"protocol_sha256": PROTOCOL_SHA256, "protocol": FROZEN_PROTOCOL},
    )

print("Frozen protocol SHA256:", PROTOCOL_SHA256)
for benchmark in BENCHMARK_ORDER:
    print(f"{benchmark:10s}: {len(EXPECTED_IDS[benchmark]):5d} samples")


## 8. Model loader

Each model is resolved from its Kaggle Input, loaded through the harness Hugging Face wrapper
(which uses `AutoTokenizer` and `AutoModelForCausalLM`), and distributed with Accelerate's
automatic device map. No quantization or weight transformation is requested. Flash Attention
is not required. The model is put in evaluation mode and gradients stay disabled.


In [ ]:
from lm_eval.models.huggingface import HFLM


def resolve_model_path(configured_path):
    path = Path(configured_path).expanduser().resolve()
    if not path.is_dir():
        raise FileNotFoundError(f"Attached model directory not found: {path}")
    if (path / "config.json").is_file():
        return path
    candidates = sorted({item.parent for item in path.rglob("config.json")})
    if len(candidates) == 1:
        warnings.warn(f"Resolved nested Hugging Face model directory: {candidates[0]}")
        return candidates[0]
    raise FileNotFoundError(
        f"Expected config.json at {path}, or exactly one nested model; found {len(candidates)} candidates."
    )


def select_chat_template(lm, model_config):
    requested = bool(model_config.get("apply_chat_template", APPLY_CHAT_TEMPLATE))
    raw_template = getattr(lm.tokenizer, "chat_template", None)
    selector = model_config.get("chat_template", True)
    if not requested:
        return False, None, raw_template, "disabled_by_configuration"
    if not raw_template:
        warnings.warn(
            f"{model_config['name']}: tokenizer has no official chat template; "
            "chat templating is explicitly disabled for this model. No fallback template is invented."
        )
        return False, None, raw_template, "missing_official_tokenizer_template"
    if isinstance(raw_template, dict) and selector is True and "default" not in raw_template:
        warnings.warn(
            f"{model_config['name']}: tokenizer exposes multiple templates without a default. "
            "Set the optional per-model 'chat_template' name to apply one; templating is disabled for now."
        )
        return False, None, raw_template, "multiple_templates_without_selected_default"
    selected = lm.chat_template(selector)
    if not selected:
        warnings.warn(f"{model_config['name']}: harness could not select an official chat template.")
        return False, None, raw_template, "harness_returned_no_template"
    # Make named/dict selection explicit for tokenizer.apply_chat_template.
    lm.tokenizer.chat_template = selected
    return True, selected, raw_template, "official_tokenizer_template"


def load_model(model_config):
    resolved_path = resolve_model_path(model_config["path"])
    batch_size = model_config.get("batch_size", DEFAULT_BATCH_SIZE)
    max_batch_size = int(model_config.get("max_batch_size", DEFAULT_MAX_BATCH_SIZE))
    print(f"Loading {model_config['name']} from {resolved_path}")
    seed_everything(SEED)
    lm = HFLM(
        pretrained=str(resolved_path),
        tokenizer=str(resolved_path),
        backend="causal",
        revision="main",
        device="cuda",
        dtype=MODEL_DTYPE,
        batch_size=batch_size,
        max_batch_size=max_batch_size,
        parallelize=True,
        trust_remote_code=bool(model_config.get("trust_remote_code", False)),
        use_fast_tokenizer=True,
    )
    lm.model.eval()
    torch.set_grad_enabled(False)
    chat_applied, selected_template, raw_template, template_status = select_chat_template(lm, model_config)
    return lm, resolved_path, {
        "requested": bool(model_config.get("apply_chat_template", APPLY_CHAT_TEMPLATE)),
        "applied": chat_applied,
        "status": template_status,
        "selected_template": selected_template,
        "raw_tokenizer_chat_template": jsonable(raw_template),
        "sha256": sha256_text(selected_template) if selected_template else None,
    }


## 9. Multiple-choice inference and stable records

MMLU, GPQA Main, and TruthfulQA MC2 are evaluated as conditional-likelihood requests.
Every choice receives its unrounded total log-likelihood and greedy-match flag. Choice text,
exact continuation, ordering, character/byte lengths, labels, and references are preserved.
No answer letter is generated and no accuracy is calculated.


In [ ]:
def choice_texts_for(benchmark, task, doc):
    if benchmark == "mmlu":
        return list(doc["choices"])
    if benchmark == "gpqa":
        return [doc[f"choice{index}"] for index in range(1, 5)]
    if benchmark == "truthfulqa":
        return list(doc["mc2_targets"]["choices"])
    raise KeyError(benchmark)


def reference_for(benchmark, task, doc, labels, texts):
    if benchmark == "mmlu":
        target_index = int(doc["answer"])
        return {
            "ground_truth_index": target_index,
            "ground_truth_label": labels[target_index],
            "ground_truth_text": texts[target_index],
        }
    if benchmark == "gpqa":
        target_label = str(task.doc_to_target(doc))
        target_index = labels.index(target_label)
        return {
            "ground_truth_index": target_index,
            "ground_truth_label": target_label,
            "ground_truth_text": texts[target_index],
            "original_correct_answer": doc.get("Correct Answer"),
            "shuffled_choice_mapping": [
                {
                    "index": index,
                    "label": labels[index],
                    "text": text,
                    "is_correct": index == target_index,
                }
                for index, text in enumerate(texts)
            ],
        }
    if benchmark == "truthfulqa":
        truth_labels = [int(item) for item in doc["mc2_targets"]["labels"]]
        return {
            "candidate_truth_labels": truth_labels,
            "true_choice_indices": [i for i, value in enumerate(truth_labels) if value == 1],
            "false_choice_indices": [i for i, value in enumerate(truth_labels) if value == 0],
            "mc2_normalization": "softmax over all saved raw candidate log-likelihoods",
        }
    raise KeyError(benchmark)


def build_common_record(model_context, benchmark, output, doc_id, prompt, reference, prediction, request_config):
    task = output.task
    task_version = (task.config.metadata or {}).get("version", output.version)
    subject = output.task_name.removeprefix("mmlu_") if benchmark == "mmlu" else None
    source_dataset = task.eval_docs
    return {
        "suite": SUITE_NAME,
        "protocol_sha256": PROTOCOL_SHA256,
        "model_name": model_context["model_name"],
        "model_path": model_context["model_path"],
        "benchmark": benchmark,
        "task_name": output.task_name,
        "task_version": task_version,
        "sample_id": sample_id_for(output.task_name, doc_id),
        "subject": subject,
        "num_fewshot": BENCHMARK_PROTOCOLS[benchmark]["num_fewshot"],
        "prompt": prompt,
        "reference": jsonable(reference),
        "prediction": jsonable(prediction),
        "generation_config": jsonable(request_config),
        "metadata": {
            "doc_id": int(doc_id),
            "source_document": jsonable(task.eval_docs[int(doc_id)]),
            "lm_eval_task_config": task.config.to_dict(),
            "dataset_fingerprint": getattr(source_dataset, "_fingerprint", None),
            "dataset_revision": BENCHMARK_PROTOCOLS[benchmark]["dataset_revision"],
            "apply_chat_template": model_context["chat_template"]["applied"],
            "chat_template_sha256": model_context["chat_template"]["sha256"],
            "fewshot_as_multiturn": model_context["chat_template"]["applied"],
            "lm_eval_version": LM_EVAL_VERSION,
            "lm_eval_commit": LM_EVAL_COMMIT,
            "created_at": utc_now(),
        },
    }


def build_mc_record(model_context, benchmark, output, doc_id, instances, responses):
    task = output.task
    doc = instances[0].doc
    labels = [str(item) for item in task.doc_to_choice(doc)]
    texts = choice_texts_for(benchmark, task, doc)
    if not (len(labels) == len(texts) == len(instances) == len(responses)):
        raise RuntimeError(f"Choice/request cardinality mismatch for {output.task_name}:{doc_id}")
    prompt = instances[0].args[0]
    if any(instance.args[0] != prompt for instance in instances):
        raise RuntimeError("Expected one shared multiple-choice prompt per document.")
    prediction_choices = []
    for index, (label, text, instance, response) in enumerate(zip(labels, texts, instances, responses)):
        loglikelihood, is_greedy = response
        continuation = instance.args[1]
        prediction_choices.append(
            {
                "index": index,
                "label": label,
                "text": text,
                "continuation": continuation,
                "loglikelihood": float(loglikelihood),
                "is_greedy": bool(is_greedy),
                "harness_choice_char_length": len(label),
                "choice_text_char_length": len(text),
                "choice_text_byte_length": len(text.encode("utf-8")),
                "continuation_char_length": len(continuation),
            }
        )
    return build_common_record(
        model_context=model_context,
        benchmark=benchmark,
        output=output,
        doc_id=doc_id,
        prompt=prompt,
        reference=reference_for(benchmark, task, doc, labels, texts),
        prediction={"choices": prediction_choices},
        request_config={
            "request_type": "loglikelihood",
            "target_delimiter": task.config.target_delimiter,
            "choice_order": labels,
        },
    )


## 10. Generation inference and stable records

GSM8K, HumanEval, and IFEval use deterministic `generate_until` requests. The returned model
continuation is stored unchanged as `prediction.raw_text` together with the exact request
parameters. IFEval's untemplated source prompt is separately retained even when the model's
official chat template changes the rendered model input.


In [ ]:
def generation_reference(benchmark, doc):
    if benchmark == "gsm8k":
        return {"answer": doc.get("answer"), "question": doc.get("question")}
    if benchmark == "humaneval":
        return {
            "task_id": doc.get("task_id"),
            "entry_point": doc.get("entry_point"),
            "test": doc.get("test"),
            "canonical_solution": doc.get("canonical_solution"),
            "execution_performed": False,
        }
    if benchmark == "ifeval":
        return {
            "original_prompt": doc.get("prompt"),
            "key": doc.get("key"),
            "instruction_id_list": doc.get("instruction_id_list"),
            "kwargs": doc.get("kwargs"),
        }
    raise KeyError(benchmark)


def build_generation_record(model_context, benchmark, output, doc_id, instance, response):
    doc = instance.doc
    prompt, request_kwargs = instance.args
    frozen_kwargs = BENCHMARK_PROTOCOLS[benchmark]["generation_kwargs"]
    if request_kwargs != frozen_kwargs:
        raise RuntimeError(
            f"Harness generation kwargs drifted for {benchmark}: {request_kwargs!r} != {frozen_kwargs!r}"
        )
    return build_common_record(
        model_context=model_context,
        benchmark=benchmark,
        output=output,
        doc_id=doc_id,
        prompt=prompt,
        reference=generation_reference(benchmark, doc),
        prediction={"raw_text": response},
        request_config={
            **request_kwargs,
            "request_type": "generate_until",
            "num_return_sequences": 1,
            "harness_stop_sequences_removed_from_returned_text": bool(request_kwargs.get("until")),
        },
    )


## 11. Resume, integrity, and benchmark runners

A benchmark is skipped only when `DONE`, its exact expected sample-ID set, JSONL integrity,
protocol hash, and file SHA256 all agree. Partial files resume at missing sample IDs. The
runner writes after small chunks and never invokes any task's scoring function.


In [ ]:
def benchmark_paths(model_dir, benchmark):
    directory = Path(model_dir) / benchmark
    return {
        "dir": directory,
        "samples": directory / "samples.jsonl",
        "done": directory / "DONE",
        "incomplete": directory / "INCOMPLETE.json",
    }


def validate_records(model_context, benchmark, records):
    ids = [record.get("sample_id") for record in records]
    id_set = set(ids)
    expected = EXPECTED_IDS[benchmark]
    unknown = id_set - expected
    if unknown:
        raise RuntimeError(f"{benchmark} contains {len(unknown)} unknown sample IDs; refusing to overwrite.")
    for record in records:
        if record.get("suite") != SUITE_NAME or record.get("protocol_sha256") != PROTOCOL_SHA256:
            raise RuntimeError(f"{benchmark} contains records from a different suite/protocol.")
        if record.get("model_name") != model_context["model_name"]:
            raise RuntimeError(f"{benchmark} contains records from a different model name.")
        if Path(record.get("model_path", "")).resolve() != Path(model_context["model_path"]).resolve():
            raise RuntimeError(f"{benchmark} contains records from a different model path.")
        prediction = record.get("prediction", {})
        if BENCHMARK_PROTOCOLS[benchmark]["output_type"] == "multiple_choice":
            choices = prediction.get("choices")
            if not choices or any("loglikelihood" not in choice for choice in choices):
                raise RuntimeError(f"{benchmark} has an incomplete likelihood record: {record.get('sample_id')}")
        elif "raw_text" not in prediction:
            raise RuntimeError(f"{benchmark} has an incomplete generation: {record.get('sample_id')}")
    return {
        "valid": id_set == expected and len(ids) == len(expected),
        "samples": len(records),
        "expected": len(expected),
        "missing": len(expected - id_set),
    }


def benchmark_is_complete(model_context, benchmark):
    paths = benchmark_paths(model_context["model_dir"], benchmark)
    if not paths["done"].is_file() or not paths["samples"].is_file():
        return False, None
    records = repair_and_read_jsonl(paths["samples"])
    validation = validate_records(model_context, benchmark, records)
    if not validation["valid"]:
        return False, validation
    done = json.loads(paths["done"].read_text(encoding="utf-8"))
    digest = sha256_file(paths["samples"])
    valid_done = (
        done.get("protocol_sha256") == PROTOCOL_SHA256
        and done.get("samples") == len(records)
        and done.get("sha256") == digest
    )
    return valid_done, {**validation, "sha256": digest}


def prepare_requests(output, model_context):
    task = output.task
    task.set_fewshot_seed(SEED)
    task._config.num_fewshot = BENCHMARK_PROTOCOLS[model_context["benchmark"]]["num_fewshot"]
    generation_kwargs = BENCHMARK_PROTOCOLS[model_context["benchmark"]]["generation_kwargs"]
    if generation_kwargs is not None:
        task._config.generation_kwargs = deepcopy(generation_kwargs)
    apply_chat = model_context["chat_template"]["applied"]
    task.build_all_requests(
        rank=0,
        world_size=1,
        cache_requests=False,
        rewrite_requests_cache=False,
        system_instruction=None,
        apply_chat_template=apply_chat,
        fewshot_as_multiturn=apply_chat,
        chat_template=model_context["lm"].apply_chat_template if apply_chat else None,
        tokenizer_name=model_context["lm"].tokenizer_name if apply_chat else "",
    )
    grouped = defaultdict(list)
    for instance in task.instances:
        grouped[int(instance.doc_id)].append(instance)
    return [(doc_id, grouped[doc_id]) for doc_id in sorted(grouped)]


@torch.inference_mode()
def run_benchmark(model_context, benchmark):
    model_context = {**model_context, "benchmark": benchmark}
    paths = benchmark_paths(model_context["model_dir"], benchmark)
    paths["dir"].mkdir(parents=True, exist_ok=True)
    complete, validation = benchmark_is_complete(model_context, benchmark)
    if complete:
        print(f"SKIP {model_context['model_name']} / {benchmark}: DONE passed integrity checks.")
        return validation

    records = repair_and_read_jsonl(paths["samples"])
    validation = validate_records(model_context, benchmark, records)
    completed_ids = {record["sample_id"] for record in records}
    print(
        f"RUN  {model_context['model_name']} / {benchmark}: "
        f"{len(completed_ids)}/{len(EXPECTED_IDS[benchmark])} already complete"
    )
    display_gpu_memory()
    if paths["incomplete"].exists():
        paths["incomplete"].unlink()

    benchmark_progress = tqdm(
        total=len(EXPECTED_IDS[benchmark]),
        initial=len(completed_ids),
        desc=f"{model_context['model_name']} | {benchmark}",
        unit="sample",
    )
    try:
        for output in TASK_OUTPUTS_BY_BENCHMARK[benchmark]:
            request_groups = prepare_requests(output, model_context)
            pending = [
                (doc_id, instances)
                for doc_id, instances in request_groups
                if sample_id_for(output.task_name, doc_id) not in completed_ids
            ]
            for group_chunk in chunks(pending, SAMPLE_CHUNK_SIZE):
                if BENCHMARK_PROTOCOLS[benchmark]["output_type"] == "multiple_choice":
                    flat_requests = [instance for _doc_id, group in group_chunk for instance in group]
                    flat_responses = model_context["lm"].loglikelihood(flat_requests, disable_tqdm=True)
                    new_records = []
                    offset = 0
                    for doc_id, instances in group_chunk:
                        count = len(instances)
                        responses = flat_responses[offset : offset + count]
                        offset += count
                        new_records.append(
                            build_mc_record(model_context, benchmark, output, doc_id, instances, responses)
                        )
                else:
                    flat_requests = [group[0] for _doc_id, group in group_chunk]
                    if any(len(group) != 1 for _doc_id, group in group_chunk):
                        raise RuntimeError(f"Expected one generation request per {benchmark} sample.")
                    responses = model_context["lm"].generate_until(flat_requests, disable_tqdm=True)
                    new_records = [
                        build_generation_record(model_context, benchmark, output, doc_id, group[0], response)
                        for (doc_id, group), response in zip(group_chunk, responses)
                    ]
                append_records(paths["samples"], new_records)
                completed_ids.update(record["sample_id"] for record in new_records)
                benchmark_progress.update(len(new_records))
            output.task._instances = []
    finally:
        benchmark_progress.close()

    records = repair_and_read_jsonl(paths["samples"])
    validation = validate_records(model_context, benchmark, records)
    if not validation["valid"]:
        atomic_write_json(
            paths["incomplete"],
            {**validation, "protocol_sha256": PROTOCOL_SHA256, "updated_at": utc_now()},
        )
        raise RuntimeError(f"{benchmark} is incomplete after generation: {validation}")

    digest = sha256_file(paths["samples"])
    done_payload = {
        "status": "complete",
        "benchmark": benchmark,
        "samples": len(records),
        "expected_samples": len(EXPECTED_IDS[benchmark]),
        "sha256": digest,
        "protocol_sha256": PROTOCOL_SHA256,
        "completed_at": utc_now(),
    }
    atomic_write_json(paths["done"], done_payload)
    return {**validation, "sha256": digest}


## 12. Model metadata and main sequential loop

Models run one at a time. Results are committed benchmark-by-benchmark. After each model,
model/tokenizer objects are deleted, garbage collection runs, and every CUDA cache is emptied
before the next checkpoint is loaded.


In [ ]:
PIP_FREEZE = subprocess.check_output(
    [sys.executable, "-m", "pip", "freeze"], text=True
).splitlines()


def model_metadata(model_config, model_context):
    lm = model_context["lm"]
    model = lm.model
    config_dict = lm.config.to_dict() if hasattr(lm.config, "to_dict") else jsonable(lm.config)
    parameter_count = model.num_parameters() if hasattr(model, "num_parameters") else sum(
        parameter.numel() for parameter in model.parameters()
    )
    return {
        "suite": SUITE_NAME,
        "protocol_sha256": PROTOCOL_SHA256,
        "model_display_name": model_config["name"],
        "configured_path": model_config["path"],
        "absolute_kaggle_path": model_context["model_path"],
        "model_config": config_dict,
        "architecture": config_dict.get("architectures", []),
        "parameter_count": int(parameter_count),
        "dtype": str(getattr(model, "dtype", MODEL_DTYPE)),
        "hf_device_map": jsonable(getattr(model, "hf_device_map", None)),
        "tokenizer_name_or_path": str(getattr(lm.tokenizer, "name_or_path", model_context["model_path"])),
        "tokenizer_class": type(lm.tokenizer).__name__,
        "chat_template": model_context["chat_template"],
        "package_versions": PACKAGE_VERSIONS,
        "pip_freeze": PIP_FREEZE,
        "gpu": GPU_INFO,
        "seed": SEED,
        "suite_version": SUITE_NAME,
        "benchmark_configs": BENCHMARK_PROTOCOLS,
        "resolved_harness_tasks": RESOLVED_TASKS,
        "generation_kwargs": {
            name: config["generation_kwargs"]
            for name, config in BENCHMARK_PROTOCOLS.items()
            if config["generation_kwargs"] is not None
        },
        "lm_eval_version": LM_EVAL_VERSION,
        "lm_eval_commit": LM_EVAL_COMMIT,
        "created_at": utc_now(),
        "scoring_performed": False,
        "human_eval_code_executed": False,
    }


def write_or_validate_model_metadata(path, payload):
    path = Path(path)
    if path.exists():
        existing = json.loads(path.read_text(encoding="utf-8"))
        identity_fields = [
            "protocol_sha256",
            "model_display_name",
            "absolute_kaggle_path",
            "lm_eval_commit",
        ]
        mismatches = [field for field in identity_fields if existing.get(field) != payload.get(field)]
        existing_chat_hash = existing.get("chat_template", {}).get("sha256")
        new_chat_hash = payload.get("chat_template", {}).get("sha256")
        if existing_chat_hash != new_chat_hash:
            mismatches.append("chat_template.sha256")
        if mismatches:
            raise RuntimeError(
                f"Existing metadata differs in {mismatches}; choose a new model name/OUTPUT_ROOT."
            )
        return existing
    atomic_write_json(path, payload)
    return payload


MANIFEST_PATH = OUTPUT_ROOT_PATH / "manifest.json"


def load_manifest():
    if MANIFEST_PATH.exists():
        manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
        if manifest.get("protocol_sha256") != PROTOCOL_SHA256:
            raise RuntimeError("Existing manifest belongs to a different frozen protocol.")
        return manifest
    return {
        "suite": SUITE_NAME,
        "protocol_sha256": PROTOCOL_SHA256,
        "created_at": utc_now(),
        "models": [],
        "entries": [],
    }


MANIFEST = load_manifest()


def update_manifest(model_name, benchmark, status, samples, digest, file_path, error=None):
    MANIFEST["entries"] = [
        entry
        for entry in MANIFEST["entries"]
        if not (entry.get("model") == model_name and entry.get("benchmark") == benchmark)
    ]
    entry = {
        "model": model_name,
        "benchmark": benchmark,
        "status": status,
        "samples": int(samples),
        "sha256": digest,
        "file": str(Path(file_path).relative_to(OUTPUT_ROOT_PATH)),
    }
    if error:
        entry["error"] = error
    MANIFEST["entries"].append(entry)
    MANIFEST["entries"].sort(key=lambda item: (item["model"], BENCHMARK_ORDER.index(item["benchmark"])))
    MANIFEST["updated_at"] = utc_now()
    atomic_write_json(MANIFEST_PATH, MANIFEST)


def validate_model_configuration():
    if not MODELS:
        raise ValueError("MODELS is empty. Add at least one attached Hugging Face model in section 1.")
    seen = set()
    for item in MODELS:
        if not isinstance(item, dict) or not item.get("name") or not item.get("path"):
            raise ValueError(f"Each MODELS entry needs non-empty name/path fields: {item!r}")
        if "<MODEL_" in item["name"] or "<MODEL_" in item["path"]:
            raise ValueError("Replace the <MODEL_NAME_...>/<MODEL_PATH_...> placeholders in section 1.")
        slug = safe_slug(item["name"])
        if slug in seen:
            raise ValueError(f"Two model names map to the same output directory: {slug}")
        seen.add(slug)


validate_model_configuration()
for configured_model in MODELS:
    model_name = configured_model["name"]
    model_slug = safe_slug(model_name)
    model_dir = OUTPUT_ROOT_PATH / model_slug
    model_dir.mkdir(parents=True, exist_ok=True)
    resolved_model_path = resolve_model_path(configured_model["path"])
    pre_context = {
        "model_name": model_name,
        "model_path": str(resolved_model_path),
        "model_dir": str(model_dir),
    }
    if model_name not in MANIFEST["models"]:
        MANIFEST["models"].append(model_name)
        atomic_write_json(MANIFEST_PATH, MANIFEST)

    already_complete = True
    for benchmark in BENCHMARK_ORDER:
        done, info = benchmark_is_complete(pre_context, benchmark)
        if not done:
            already_complete = False
            break
    if already_complete:
        print(f"SKIP MODEL {model_name}: every benchmark passed DONE integrity checks.")
        if model_name not in MANIFEST["models"]:
            MANIFEST["models"].append(model_name)
        for benchmark in BENCHMARK_ORDER:
            paths = benchmark_paths(model_dir, benchmark)
            records = repair_and_read_jsonl(paths["samples"])
            update_manifest(
                model_name,
                benchmark,
                "complete",
                len(records),
                sha256_file(paths["samples"]),
                paths["samples"],
            )
        continue

    lm = None
    try:
        lm, resolved_model_path, chat_metadata = load_model(configured_model)
        context = {
            "lm": lm,
            "model_name": model_name,
            "model_path": str(resolved_model_path),
            "model_dir": str(model_dir),
            "chat_template": chat_metadata,
        }
        metadata = model_metadata(configured_model, context)
        write_or_validate_model_metadata(model_dir / "metadata.json", metadata)
        for benchmark in BENCHMARK_ORDER:
            paths = benchmark_paths(model_dir, benchmark)
            try:
                result = run_benchmark(context, benchmark)
                update_manifest(
                    model_name,
                    benchmark,
                    "complete",
                    result["samples"],
                    result["sha256"],
                    paths["samples"],
                )
            except Exception as exc:
                records = repair_and_read_jsonl(paths["samples"])
                error_payload = {
                    "status": "incomplete",
                    "error_type": type(exc).__name__,
                    "error": str(exc),
                    "samples": len(records),
                    "expected_samples": len(EXPECTED_IDS[benchmark]),
                    "updated_at": utc_now(),
                }
                atomic_write_json(paths["incomplete"], error_payload)
                update_manifest(
                    model_name,
                    benchmark,
                    "incomplete",
                    len(records),
                    sha256_file(paths["samples"]) if paths["samples"].exists() else None,
                    paths["samples"],
                    error=f"{type(exc).__name__}: {exc}",
                )
                print(f"ERROR {model_name} / {benchmark}: {type(exc).__name__}: {exc}")
                traceback.print_exc()
    except Exception as exc:
        print(f"MODEL ERROR {model_name}: {type(exc).__name__}: {exc}")
        traceback.print_exc()
        for benchmark in BENCHMARK_ORDER:
            paths = benchmark_paths(model_dir, benchmark)
            done, info = benchmark_is_complete(pre_context, benchmark)
            if done:
                continue
            records = repair_and_read_jsonl(paths["samples"])
            paths["dir"].mkdir(parents=True, exist_ok=True)
            error_payload = {
                "status": "incomplete",
                "error_type": type(exc).__name__,
                "error": str(exc),
                "samples": len(records),
                "expected_samples": len(EXPECTED_IDS[benchmark]),
                "updated_at": utc_now(),
            }
            atomic_write_json(paths["incomplete"], error_payload)
            update_manifest(
                model_name,
                benchmark,
                "incomplete",
                len(records),
                sha256_file(paths["samples"]) if paths["samples"].exists() else None,
                paths["samples"],
                error=f"{type(exc).__name__}: {exc}",
            )
    finally:
        if lm is not None:
            try:
                del lm._model
            except Exception:
                pass
            del lm
        gc.collect()
        torch.cuda.empty_cache()
        for gpu_index in range(torch.cuda.device_count()):
            with torch.cuda.device(gpu_index):
                torch.cuda.empty_cache()
        print(f"Released model resources for {model_name}.")


## 13. Final validation

This pass re-reads every JSONL file, checks the exact sample-ID inventory and raw prediction
fields, verifies `DONE` hashes, and refreshes the manifest. Incomplete benchmarks remain
clearly marked and can be resumed by rerunning the notebook with the same outputs attached or
restored under `OUTPUT_ROOT`.


In [ ]:
FINAL_VALIDATION = []
for configured_model in MODELS:
    model_name = configured_model["name"]
    model_dir = OUTPUT_ROOT_PATH / safe_slug(model_name)
    resolved_model_path = resolve_model_path(configured_model["path"])
    context = {
        "model_name": model_name,
        "model_path": str(resolved_model_path),
        "model_dir": str(model_dir),
    }
    for benchmark in BENCHMARK_ORDER:
        paths = benchmark_paths(model_dir, benchmark)
        complete, info = benchmark_is_complete(context, benchmark)
        row = {
            "model": model_name,
            "benchmark": benchmark,
            "status": "complete" if complete else "incomplete",
            "samples": (info or {}).get("samples", 0),
            "expected": len(EXPECTED_IDS[benchmark]),
        }
        FINAL_VALIDATION.append(row)
        print(
            f"{row['model']} | {row['benchmark']}: {row['status']} "
            f"({row['samples']}/{row['expected']})"
        )
atomic_write_json(OUTPUT_ROOT_PATH / "validation.json", FINAL_VALIDATION)


## 14. Final manifest

`manifest.json` contains one row per model/benchmark with status, count, SHA256, and relative
file path. It contains no computed benchmark metrics.


In [ ]:
MANIFEST = load_manifest()
MANIFEST["validation"] = FINAL_VALIDATION
MANIFEST["scoring_performed"] = False
MANIFEST["human_eval_code_executed"] = False
MANIFEST["finalized_at"] = utc_now()
atomic_write_json(MANIFEST_PATH, MANIFEST)
print(json.dumps(MANIFEST, indent=2))


## 15. ZIP export

The complete generation directory is archived for download. The ZIP includes the frozen
protocol, model metadata, every JSONL sample file, completion markers, validation, and the
manifest—never model weights or Hugging Face caches.


In [ ]:
ZIP_PATH = OUTPUT_ROOT_PATH.parent / f"{OUTPUT_ROOT_PATH.name}.zip"
temporary_base = OUTPUT_ROOT_PATH.parent / f".{OUTPUT_ROOT_PATH.name}_export"
temporary_zip = Path(str(temporary_base) + ".zip")
if temporary_zip.exists():
    temporary_zip.unlink()
shutil.make_archive(
    str(temporary_base),
    "zip",
    root_dir=OUTPUT_ROOT_PATH.parent,
    base_dir=OUTPUT_ROOT_PATH.name,
)
os.replace(temporary_zip, ZIP_PATH)
print("=" * 80)
print(f"DOWNLOAD THIS FILE: {ZIP_PATH}")
print(f"ZIP SHA256: {sha256_file(ZIP_PATH)}")
print("=" * 80)
